# How TMDB Movie VAD + Emotion Scores Are Built

This notebook documents the scoring pipeline that produced the **[TMDB Movie VAD + Emotion Intensity Scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores)** dataset.

**Input:** TMDB Movies Dataset — plot keywords field  
**Output:** 15 affective columns per movie (3 keyword groups + 12 VAD/emotion scores)

### Pipeline

```
TMDB keywords (comma-separated phrases)
  → tokenize on whitespace / hyphens
  → lemmatize (noun / verb / adj POS — maximises lexicon coverage)
  → look up each lemma in NRC VAD + Emotion Intensity dicts
  → average matched token scores per keyword
  → average keyword scores per movie
  → bucket each keyword by its individual valence score
```

### Lexicons (Saif M. Mohammad, NRC Canada — CC BY-NC-SA 4.0)

| Lexicon | Words | Output |
|---------|-------|--------|
| NRC Emotion Intensity | 5,891 | continuous 0–1 per emotion (8 emotions) |
| NRC VAD | 19,971 | valence, arousal, dominance 0–1 |

In [ ]:
import re
import pandas as pd
from pathlib import Path

# NLTK wordnet — available on Kaggle without internet
import nltk
nltk.data.path.append('/usr/share/nltk_data')
try:
    from nltk.stem import WordNetLemmatizer
    nltk.data.find('corpora/wordnet')
    _lemmatizer = WordNetLemmatizer()
    HAS_LEMMA = True
    print('wordnet: ready')
except LookupError:
    HAS_LEMMA = False
    print('wordnet: not available — using raw tokens')

# Load TMDB
tmdb_csv = sorted(Path('/kaggle/input').rglob('*.csv'))[0]
df = pd.read_csv(tmdb_csv, low_memory=False, nrows=50_000)
print(f'Loaded {len(df):,} rows')

## Step 1 — Tokenize keywords

Keywords are comma-separated phrases. Each phrase is split on whitespace/hyphens into alpha tokens.

In [ ]:
def _tokenize(keyword: str) -> list:
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]

# Example
for kw in ['dying and death', 'psychological-thriller', 'space marine']:
    print(f'  {kw!r:30s} → {_tokenize(kw)}')

## Step 2 — Lemmatize

For each token, try noun, verb, and adjective lemmas in order. First match wins. This significantly boosts coverage — `"dying"` → `"die"`, `"coming"` → `"come"`.

In [ ]:
def _lemma_candidates(tok: str) -> list:
    if not HAS_LEMMA:
        return [tok]
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma); out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out

for tok in ['dying', 'deaths', 'paranormal', 'psychological']:
    print(f'  {tok!r:20s} → {_lemma_candidates(tok)}')

## Step 3 — NRC VAD lookup and keyword scoring

For each token, walk through its lemma candidates and take the first match in the NRC VAD dict. Unmatched keywords return NaN (not 0 — the VAD scale is 0–1 with 0.5 as neutral, so 0 would be falsely negative).

Keywords are then bucketed by their individual valence:
- **positive_keywords**: valence ≥ 0.5
- **negative_keywords**: valence < 0.5
- **unknown_keywords**: no VAD entry

In [ ]:
# The real pipeline loads NRC VAD from a file. Here we show the lookup logic:

def score_keyword(keyword, vad_lookup):
    """Score a keyword phrase against a VAD lookup dict."""
    tokens = _tokenize(keyword)
    v_rows = []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in vad_lookup:
                v_rows.append(vad_lookup[lemma])
                break  # first match per token
    if not v_rows:
        return {d: float('nan') for d in ('valence', 'arousal', 'dominance')}
    return {d: sum(r[d] for r in v_rows) / len(v_rows) for d in ('valence', 'arousal', 'dominance')}

def classify_keywords(keywords_str, vad_lookup):
    """Bucket a movie's keywords by valence."""
    if not isinstance(keywords_str, str):
        return {'positive_keywords': '', 'negative_keywords': '', 'unknown_keywords': ''}
    pos, neg, unk = [], [], []
    for kw in [k.strip() for k in keywords_str.split(',') if k.strip()]:
        v = score_keyword(kw, vad_lookup).get('valence', float('nan'))
        if v != v:      unk.append(kw)
        elif v >= 0.5:  pos.append(kw)
        else:           neg.append(kw)
    return {'positive_keywords': ', '.join(pos), 'negative_keywords': ', '.join(neg), 'unknown_keywords': ', '.join(unk)}

print('Lookup logic defined — see full pipeline at:')
print('https://github.com/bdelanghe/imdb-kaggle/blob/main/02_score_and_export.ipynb')

## Real scores — The Sixth Sense & Alien

The actual scores (from the full NRC VAD lexicon with 19,971 words):

In [ ]:
# Real output from the scored dataset
# https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores

sixth_sense = {
    'title':             'The Sixth Sense',
    'keywords':          'dying and death, child abuse, philadelphia, pennsylvania, sense of guilt, afterlife, marriage crisis, loss of loved one, confidence, psychology, paranormal phenomena, single, psychological thriller, cowardliness, single mother, school play, ghost, child, spiritism, ghost child',
    'positive_keywords': 'pennsylvania, afterlife, marriage crisis, loss of loved one, confidence, psychology, single mother, school play, child, ghost child',
    'negative_keywords': 'dying and death, child abuse, sense of guilt, paranormal phenomena, single, psychological thriller, cowardliness, ghost, spiritism',
    'unknown_keywords':  'philadelphia',
    'valence': -0.041, 'arousal': 0.551, 'dominance': 0.511,
    'sentiment': 'negative',
    'fear': 0.424, 'sadness': 0.463, 'trust': 0.556, 'disgust': 0.456,
}

alien = {
    'title':             'Alien',
    'keywords':          'android, spacecraft, space marine, biology, dystopia, countdown, space suit, beheading, space travel, cowardice, alien, space, female protagonist, parasite, space opera, cosmos, xenomorph',
    'positive_keywords': 'android, spacecraft, space marine, space suit, space travel, space, female protagonist, space opera, cosmos',
    'negative_keywords': 'biology, dystopia, countdown, beheading, cowardice, alien, parasite',
    'unknown_keywords':  'xenomorph',
    'valence': -0.004, 'arousal': 0.492, 'dominance': 0.539,
    'sentiment': 'negative',
    'fear': 0.356, 'disgust': 0.614, 'anticipation': 0.609,
}

for movie in [sixth_sense, alien]:
    print(f"Title:    {movie['title']}")
    print(f"Valence:  {movie['valence']:.3f}  ({movie['sentiment']})")
    print(f"positive: {movie['positive_keywords']}")
    print(f"negative: {movie['negative_keywords']}")
    print(f"unknown:  {movie['unknown_keywords']}")
    top = sorted([(k,v) for k,v in movie.items() if k in ('fear','sadness','trust','disgust','anticipation','joy','anger','surprise')], key=lambda x: -x[1])
    print(f"top emotions: {', '.join(f"{k} {v:.2f}" for k,v in top[:3])}")
    print()

## Coverage

```python
# 1.38M movies total
# ~23% have TMDB keywords → receive scores
# ~77% have no keywords → sentiment = 'unknown'
```

Full dataset: **[kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores)**  
Source code: **[github.com/bdelanghe/imdb-kaggle](https://github.com/bdelanghe/imdb-kaggle)**

Lexicon citation: Mohammad, S.M. (2020). [Practical and Ethical Considerations in the Effective use of Emotion and Sentiment Lexicons](https://www.saifmohammad.com/WebDocs/EmoLex-Ethics-Data-Statement.pdf). NRC Canada.  
License: CC BY-NC-SA 4.0